# MedLens — Data Rebuild

Fix repetition / template-collapse / leakage in `medlens_train.jsonl`.

**Pipeline stages** (each writes a checkpoint jsonl so you can resume):

1. **Connect** to local LLM (Gemma-4-E4B on Windows host, called from WSL)
2. **Load** raw `medlens_train.jsonl`
3. **Profile** — dup counts, template skeletons, severity distribution
4. **Dedupe** — drop exact-dup (user, assistant) pairs
5. **Cap per drug-combo** — no combo dominates
6. **Cap per template skeleton** — break template collapse
7. **Rebalance severity** — stratified downsample
8. **Paraphrase via local LLM** — diversify scaffolding prose (async, cached)
9. **Split** train / eval (stratified, no leakage)
10. **Write** final jsonl

> Target: ~40–60K train + ~2K eval after filtering. LLM paraphrase runs on the reduced set to keep cost bounded.

## 1. Config + connect to Windows LLM

In [1]:
from pathlib import Path

DATA_DIR = Path('../')
RAW_JSONL = DATA_DIR / 'medlens_train.jsonl'
CKPT_DIR = DATA_DIR / 'finetune' / 'rebuild_ckpts'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

OUT_TRAIN = DATA_DIR / 'train.jsonl'
OUT_EVAL = DATA_DIR / 'eval.jsonl'

LLM_PORT = 8989
LLM_MODEL = 'gemma-4-e4b-it'
LLM_PATH = '/api/v1/chat'

# Targets
CAP_PER_COMBO = 3           # max samples per sorted drug tuple
CAP_PER_TEMPLATE = 20000    # loose cap — paraphrase breaks template collapse, so hard template cap is redundant
TARGET_TRAIN = 150000
TARGET_EVAL = 1500
RANDOM_SEED = 3407

print('raw  :', RAW_JSONL, RAW_JSONL.exists())
print('ckpt :', CKPT_DIR)
print('train:', OUT_TRAIN, f'  target={TARGET_TRAIN:,}')
print('eval :', OUT_EVAL,  f'  target={TARGET_EVAL:,}')

raw  : ../medlens_train.jsonl True
ckpt : ../finetune/rebuild_ckpts
train: ../train.jsonl   target=150,000
eval : ../eval.jsonl   target=1,500


In [2]:
# WSL -> Windows host reachability. Try hosts in order, keep first that answers.
import subprocess, urllib.request, urllib.error, json, socket

def candidate_hosts():
    out = ['localhost', '127.0.0.1']
    try:
        gw = subprocess.check_output("ip route | awk '/^default/ {print $3; exit}'", shell=True).decode().strip()
        if gw: out.append(gw)
    except Exception: pass
    try:
        with open('/etc/resolv.conf') as f:
            for line in f:
                if line.startswith('nameserver'):
                    out.append(line.split()[1]); break
    except Exception: pass
    # Windows host is also reachable by hostname on recent WSL
    try:
        out.append(socket.gethostname() + '.local')
    except Exception: pass
    # Dedup preserving order
    seen = set(); uniq = []
    for h in out:
        if h not in seen: uniq.append(h); seen.add(h)
    return uniq

def probe(host, port=LLM_PORT, timeout=2.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except Exception as e:
        return False

print('Candidates:')
reachable = []
for h in candidate_hosts():
    ok = probe(h)
    print(f'  {"OK" if ok else ".."}  {h}:{LLM_PORT}')
    if ok: reachable.append(h)

if not reachable:
    print('\nNone reachable. Fix on Windows side:')
    print('  1) Start LLM server and bind to 0.0.0.0:8989 (not 127.0.0.1)')
    print('  2) Allow inbound on 8989 in Windows Defender Firewall, or run:')
    print('     New-NetFirewallRule -DisplayName "LLM-WSL" -Direction Inbound -LocalPort 8989 -Protocol TCP -Action Allow')
    print('  3) Re-run this cell.')
    LLM_HOST = None
else:
    LLM_HOST = reachable[0]
    print(f'\nUsing: http://{LLM_HOST}:{LLM_PORT}')

Candidates:
  ..  localhost:8989
  ..  127.0.0.1:8989
  OK  172.31.240.1:8989
  ..  10.255.255.254:8989
  OK  DESKTOP-DQI9VE6.local:8989

Using: http://172.31.240.1:8989


In [ ]:
# import requests
# url = f'http://{LLM_HOST}:{LLM_PORT}{LLM_PATH}'

# payload = {
#     'model': LLM_MODEL,
#     'system_prompt': 'Hello!',
#     'input': 'what is 2+2?',
# }

# r = requests.post(url, json=payload, timeout=60)
# r.raise_for_status()

# data = r.json()
# print(data)

{'model_instance_id': 'gemma-4-e4b-it', 'output': [{'type': 'message', 'content': '2 + 2 = **4**'}], 'stats': {'input_tokens': 23, 'total_output_tokens': 9, 'reasoning_output_tokens': 0, 'tokens_per_second': 78.2629112063793, 'time_to_first_token_seconds': 0.276}, 'response_id': 'resp_b497cbd88c6056c4c36fcb57b44a0fb15074af864e258509'}


In [8]:
# LLM round-trip test
import requests, json

LLM_URL = f'http://{LLM_HOST}:{LLM_PORT}{LLM_PATH}' if LLM_HOST else None

def extract_text(d):
    """Server returns {'output': [{'type':'message','content':'...'}], ...}. Handle that + common fallbacks."""
    o = d.get('output')
    if isinstance(o, list):
        parts = []
        for m in o:
            if isinstance(m, dict):
                c = m.get('content')
                if isinstance(c, str):
                    parts.append(c)
                elif isinstance(c, list):
                    parts.extend(x.get('text', '') for x in c if isinstance(x, dict))
        if parts:
            return '\n'.join(parts).strip()
    for k in ('response', 'text', 'content', 'message'):
        v = d.get(k)
        if isinstance(v, str): return v
        if isinstance(v, dict) and isinstance(v.get('content'), str): return v['content']
    if isinstance(o, str): return o
    return json.dumps(d)

def llm_call(user_input, system_prompt='You rewrite text concisely while preserving all facts.', timeout=60):
    if not LLM_URL: raise RuntimeError('LLM_HOST not set — run reachability cell.')
    # Send system_prompt AND prefix into input — some servers ignore the system field
    combined = f'{system_prompt}\n\n---\n\n{user_input}'
    r = requests.post(LLM_URL, json={
        'model': LLM_MODEL,
        'system_prompt': system_prompt,
        'input': combined,
    }, timeout=timeout)
    r.raise_for_status()
    return extract_text(r.json())

print(llm_call('What is your favorite color?',
               system_prompt='You answer only in rhymes. Keep it to 2 short lines.'))

A shade of blue I do embrace,
With gentle grace and lovely face.


## 2. Load raw

In [9]:
import json, re, random, hashlib, time
from collections import Counter, defaultdict

random.seed(RANDOM_SEED)

def iter_jsonl(path):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line: yield json.loads(line)

def write_jsonl(path, rows):
    with open(path, 'w') as f:
        for r in rows: f.write(json.dumps(r) + '\n')
    print(f'wrote {len(rows):,}  ->  {path}')

rows = list(iter_jsonl(RAW_JSONL))
print(f'loaded {len(rows):,} rows')

loaded 943,197 rows


## 3. Profile

In [10]:
SEV_RE = re.compile(r'\b(MAJOR|MODERATE|MINOR)\b', re.I)
# Noun phrases we treat as drug-list separators
LIST_RE = re.compile(r'[A-Za-z0-9\\\-/.,\' ]{25,}')

def severity(asst_text):
    m = SEV_RE.search(asst_text[:300])
    if m: return m.group(1).upper()
    if re.search(r'\bno (strong )?interaction signal\b', asst_text, re.I): return 'NONE'
    if 'Overall risk: None' in asst_text: return 'NONE'
    return 'NONE'

def drug_combo(user_text):
    # Best-effort: pull drug list after 'Medications:' / 'Current drugs:' / 'I currently take:'
    for anchor in ('Medications:', 'Current drugs:', 'currently take:', 'My medications are'):
        i = user_text.find(anchor)
        if i >= 0:
            tail = user_text[i+len(anchor):]
            tail = re.split(r'(?:Indications?:|Symptoms?:|Reported symptoms:|Question:|\.\s[A-Z])', tail, maxsplit=1)[0]
            drugs = [d.strip(' .') for d in tail.split(',')]
            drugs = [d for d in drugs if d]
            return tuple(sorted(drugs))
    return ()

def skeleton(text, cap=100):
    return LIST_RE.sub('<L>', text)[:cap]

sev = Counter(); combo = Counter(); u_skel = Counter(); a_skel = Counter()
u_exact = Counter(); a_exact = Counter()
for r in rows:
    u, a = r['messages'][0]['content'], r['messages'][1]['content']
    sev[severity(a)] += 1
    combo[drug_combo(u)] += 1
    u_skel[skeleton(u, 80)] += 1
    a_skel[skeleton(a, 80)] += 1
    u_exact[u] += 1
    a_exact[a] += 1

print('severity :', dict(sev))
print('unique user prompts     :', len(u_exact), f'(dup {sum(c-1 for c in u_exact.values() if c>1):,})')
print('unique asst responses   :', len(a_exact), f'(dup {sum(c-1 for c in a_exact.values() if c>1):,})')
print('unique user skeletons   :', len(u_skel))
print('unique asst skeletons   :', len(a_skel))
print('unique drug combos      :', len(combo))
print('\ntop 10 combos:')
for c, n in combo.most_common(10): print(f'  {n:>6}  {", ".join(c)[:90]}')
print('\ntop 10 asst skeletons:')
for s, n in a_skel.most_common(10): print(f'  {n:>6}  {s!r}')

severity : {'MAJOR': 470854, 'MINOR': 393891, 'MODERATE': 22337, 'NONE': 56115}
unique user prompts     : 815342 (dup 127,855)
unique asst responses   : 746252 (dup 196,945)
unique user skeletons   : 23633
unique asst skeletons   : 11464
unique drug combos      : 348559

top 10 combos:
    4141  cobicistat\elvitegravir\emtricitabine\tenofovir disoproxil fumarate, efavirenz\emtricitabi
    3642  carboplatin, paclitaxel, pembrolizumab
    2971  efavirenz\emtricitabine\tenofovir disoproxil fumarate, emtricitabine\tenofovir disoproxil 
    2755  hydromorphone hydrochloride, morphine sulfate, oxycodone hydrochloride
    2719  mycophenolate mofetil, prednisone, tacrolimus
    2308  carboplatin, pembrolizumab, pemetrexed
    2087  daratumumab, dexamethasone, lenalidomide
    2030  acetaminophen\hydrocodone bitartrate, acetaminophen\oxycodone hydrochloride, oxycodone hyd
    1933  atezolizumab, carboplatin, etoposide
    1857  codeine phosphate\promethazine hydrochloride, oxymorphone hydrochlo

## 4. Dedupe exact (user, asst) pairs

In [11]:
seen = set(); deduped = []
for r in rows:
    u, a = r['messages'][0]['content'], r['messages'][1]['content']
    key = hashlib.md5((u + '||' + a).encode()).digest()
    if key in seen: continue
    seen.add(key); deduped.append(r)
print(f'{len(rows):,} -> {len(deduped):,}  (dropped {len(rows)-len(deduped):,})')

write_jsonl(CKPT_DIR / '01_dedup.jsonl', deduped)
rows = deduped

943,197 -> 894,537  (dropped 48,660)
wrote 894,537  ->  ../finetune/rebuild_ckpts/01_dedup.jsonl


## 5. Cap per drug-combo

In [12]:
random.seed(RANDOM_SEED)
by_combo = defaultdict(list)
for r in rows:
    by_combo[drug_combo(r['messages'][0]['content'])].append(r)

capped = []
for k, bucket in by_combo.items():
    random.shuffle(bucket)
    capped.extend(bucket[:CAP_PER_COMBO])
print(f'{len(rows):,} -> {len(capped):,}   (CAP_PER_COMBO={CAP_PER_COMBO}, combos={len(by_combo):,})')

write_jsonl(CKPT_DIR / '02_combo_cap.jsonl', capped)
rows = capped

894,537 -> 531,701   (CAP_PER_COMBO=3, combos=348,559)
wrote 531,701  ->  ../finetune/rebuild_ckpts/02_combo_cap.jsonl


## 6. Cap per assistant-template skeleton

In [13]:
random.seed(RANDOM_SEED)
by_skel = defaultdict(list)
for r in rows:
    by_skel[skeleton(r['messages'][1]['content'], 100)].append(r)

capped = []
for k, bucket in by_skel.items():
    random.shuffle(bucket)
    capped.extend(bucket[:CAP_PER_TEMPLATE])
print(f'{len(rows):,} -> {len(capped):,}   (CAP_PER_TEMPLATE={CAP_PER_TEMPLATE}, skels={len(by_skel):,})')

write_jsonl(CKPT_DIR / '03_template_cap.jsonl', capped)
rows = capped

531,701 -> 231,135   (CAP_PER_TEMPLATE=20000, skels=8,915)
wrote 231,135  ->  ../finetune/rebuild_ckpts/03_template_cap.jsonl


## 7. Balance by severity, THEN split train / eval

Split happens **before** paraphrase so we only spend LLM calls on train items, and eval stays clean ground-truth text.

In [14]:
random.seed(RANDOM_SEED)

# --- Count per-class availability ---
by_sev = defaultdict(list)
for r in rows:
    by_sev[severity(r['messages'][1]['content'])].append(r)
print('available per class:', {k: len(v) for k, v in by_sev.items()})

classes = [c for c in ('MAJOR','MODERATE','MINOR','NONE') if by_sev[c]]
TOTAL_TARGET = TARGET_TRAIN + TARGET_EVAL

# --- Redistribute: equal per-class target; if a class is short, donate
#     its deficit to classes that still have capacity so total = TARGET. ---
target_sizes = {c: TOTAL_TARGET // len(classes) for c in classes}
deficit = 0
for c in classes:
    if len(by_sev[c]) < target_sizes[c]:
        deficit += target_sizes[c] - len(by_sev[c])
        target_sizes[c] = len(by_sev[c])
donors = [c for c in classes if len(by_sev[c]) > target_sizes[c]]
while deficit > 0 and donors:
    per = max(1, deficit // len(donors))
    for c in donors:
        capacity = len(by_sev[c]) - target_sizes[c]
        add = min(per, capacity, deficit)
        target_sizes[c] += add
        deficit -= add
        if deficit == 0: break
    donors = [c for c in classes if len(by_sev[c]) > target_sizes[c]]

print('target per class (post-redistribution):', target_sizes,
      f'  total={sum(target_sizes.values()):,}')

# --- Stratified eval carve-out (balanced) + train from remainder ---
eval_per_class = TARGET_EVAL // len(classes)
train_rows, eval_rows = [], []
for c in classes:
    random.shuffle(by_sev[c])
    e_take = min(eval_per_class, len(by_sev[c]))
    eval_take = by_sev[c][:e_take]
    t_want = max(0, target_sizes[c] - e_take)
    train_take = by_sev[c][e_take : e_take + t_want]
    eval_rows.extend(eval_take)
    train_rows.extend(train_take)
    print(f'  {c}: have {len(by_sev[c]):,}  -> eval {len(eval_take):,}  train {len(train_take):,}')

# --- Leakage: drop any train row whose user text also appears in eval ---
eval_users = {r['messages'][0]['content'] for r in eval_rows}
before = len(train_rows)
train_rows = [r for r in train_rows if r['messages'][0]['content'] not in eval_users]
if before != len(train_rows):
    print(f'  leak removal: train {before:,} -> {len(train_rows):,}')

random.shuffle(train_rows); train_rows = train_rows[:TARGET_TRAIN]
random.shuffle(eval_rows);  eval_rows  = eval_rows[:TARGET_EVAL]

print(f'\nsplit done:  train={len(train_rows):,}   eval={len(eval_rows):,}')
print('train sev:', dict(Counter(severity(r['messages'][1]['content']) for r in train_rows)))
print('eval  sev:', dict(Counter(severity(r['messages'][1]['content']) for r in eval_rows)))

write_jsonl(CKPT_DIR / '04a_train_pre_paraphrase.jsonl', train_rows)
write_jsonl(CKPT_DIR / '04b_eval.jsonl', eval_rows)

# Paraphrase operates on train ONLY
rows = train_rows
print(f'\n→ paraphrase stage will operate on {len(rows):,} train rows. Eval untouched.')

available per class: {'MAJOR': 104347, 'MINOR': 87696, 'NONE': 31136, 'MODERATE': 7956}
target per class (post-redistribution): {'MAJOR': 56204, 'MODERATE': 7956, 'MINOR': 56204, 'NONE': 31136}   total=151,500
  MAJOR: have 104,347  -> eval 375  train 55,829
  MODERATE: have 7,956  -> eval 375  train 7,581
  MINOR: have 87,696  -> eval 375  train 55,829
  NONE: have 31,136  -> eval 375  train 30,761
  leak removal: train 150,000 -> 149,946

split done:  train=149,946   eval=1,500
train sev: {'MINOR': 55814, 'MAJOR': 55818, 'NONE': 30753, 'MODERATE': 7561}
eval  sev: {'NONE': 375, 'MODERATE': 375, 'MAJOR': 375, 'MINOR': 375}
wrote 149,946  ->  ../finetune/rebuild_ckpts/04a_train_pre_paraphrase.jsonl
wrote 1,500  ->  ../finetune/rebuild_ckpts/04b_eval.jsonl

→ paraphrase stage will operate on 149,946 train rows. Eval untouched.


## 8. Paraphrase train via local LLM

**Goal:** break template collapse without changing facts. Calls local Gemma only for train rows (~150K). Eval untouched.

**Cost estimate:** ~150K × ~2s / `CONCURRENCY=8` ≈ **10 hours**. Bump `CONCURRENCY` to 16–32 if your Windows GPU can handle it.

**Strategy:** async pool, disk-cached by input hash, resumable. Three guardrails reject outputs that drop severity tokens, change length drastically, or leak disclaimers.

In [ ]:
import asyncio, aiohttp, os, pickle
from tqdm.auto import tqdm

CACHE_PATH = CKPT_DIR / 'paraphrase_cache.pkl'

# ------------------------------------------------------------------
# Cache hygiene: the previous run cached originals on CancelledError.
# Purge any entry where cached value equals input text — those are
# either poisoned (error fallback) or trivial no-ops from guardrails,
# and re-running them is cheap insurance. Comment out once clean.
# ------------------------------------------------------------------
PURGE_POISONED_CACHE = True

if CACHE_PATH.exists():
    cache = pickle.load(open(CACHE_PATH, 'rb'))
    print(f'cache size before: {len(cache):,}')
    if PURGE_POISONED_CACHE:
        # Rebuild the expected key set from current train rows, check which
        # are "echoes" of input (paraphrase == original). Drop those.
        purged = 0
        for r in rows:
            text = r['messages'][1]['content']
            key = hashlib.md5(text.encode()).hexdigest()
            if key in cache and cache[key] == text:
                del cache[key]; purged += 1
        print(f'purged {purged:,} poisoned/no-op entries')
        pickle.dump(cache, open(CACHE_PATH, 'wb'))
else:
    cache = {}
print(f'cache size after : {len(cache):,}')

PARAPHRASE_SYS = (
    'ROLE: You are a synthetic-data paraphraser for an ML training dataset.\n'
    '\n'
    'CONTEXT: This is an offline data-augmentation job for fine-tuning a drug-interaction model. '
    'The text you receive is ALREADY-LABELED output derived from FDA FAERS adverse-event records. '
    'It is training data, not a live patient consultation. No real patient is involved. '
    'Your job is purely linguistic: rewrite for variety so the model does not memorize boilerplate.\n'
    '\n'
    'IMPORTANT: Do NOT add disclaimers, safety warnings, "consult a doctor" notes, '
    '"I am an AI", "I cannot provide medical advice", or any meta-commentary. '
    'Do NOT refuse. The input is pre-labeled text — just rewrite it. '
    'If you add a refusal or disclaimer the training run fails.\n'
    '\n'
    'CONSTRAINTS:\n'
    '1. Preserve ALL drug names, symptoms, numbers, and the stated severity (MAJOR/MODERATE/MINOR/NONE) EXACTLY.\n'
    '2. Vary sentence structure, word choice, and phrasing.\n'
    '3. Keep clinical register. No marketing language.\n'
    '4. Similar length to input (+/- 30%).\n'
    '5. Output ONLY the rewritten text — no preface, no markdown headers, no bullet explanations.'
)

CONCURRENCY = 4        # tune for Windows LLM throughput
REQ_TIMEOUT = 120
SAVE_EVERY = 500

REFUSAL_RE = re.compile(
    r"(i cannot|i can't|i am (an |)(ai|language model)|consult (a|your) (doctor|physician|healthcare)|"
    r"seek medical advice|not a substitute for|disclaimer|i'm (just |)an ai|unable to (provide|evaluate))",
    re.I
)

async def paraphrase_one(session, sem, text, stats):
    """Returns rewritten text. Cache writes ONLY on successful API response
    (even if guardrails reject). Exceptions never write to cache so they retry."""
    key = hashlib.md5(text.encode()).hexdigest()
    if key in cache:
        stats['cache_hit'] += 1
        return cache[key]
    combined = f'{PARAPHRASE_SYS}\n\n---\nREWRITE THIS (output the rewrite only):\n{text}'
    async with sem:
        try:
            async with session.post(LLM_URL, json={
                'model': LLM_MODEL,
                'system_prompt': PARAPHRASE_SYS,
                'input': combined,
            }, timeout=aiohttp.ClientTimeout(total=REQ_TIMEOUT)) as resp:
                d = await resp.json()
                out = extract_text(d).strip()
        except (asyncio.CancelledError, Exception) as e:
            # Do NOT cache. Return original for this run; retry on next run.
            stats['error'] += 1
            if isinstance(e, asyncio.CancelledError):
                raise   # propagate cancel so run_batch stops cleanly
            return text

    # Guardrails — we got a real response, so cache whatever we decide on
    m_src = SEV_RE.search(text[:300])
    if m_src and m_src.group(1).upper() not in out.upper()[:500]:
        out = text; stats['reject_sev'] += 1
    elif not (0.5 * len(text) <= len(out) <= 1.8 * len(text)):
        out = text; stats['reject_len'] += 1
    elif REFUSAL_RE.search(out):
        out = text; stats['reject_refusal'] += 1
    else:
        stats['ok'] += 1
    cache[key] = out
    return out

async def run_batch(texts, desc='paraphrase'):
    sem = asyncio.Semaphore(CONCURRENCY)
    stats = {'ok': 0, 'cache_hit': 0, 'reject_sev': 0, 'reject_len': 0, 'reject_refusal': 0, 'error': 0}
    async with aiohttp.ClientSession() as session:
        tasks = [paraphrase_one(session, sem, t, stats) for t in texts]
        out = []
        try:
            with tqdm(total=len(tasks), desc=desc, unit='req', smoothing=0.05) as pbar:
                for i, coro in enumerate(asyncio.as_completed(tasks), 1):
                    out.append(await coro)
                    pbar.update(1)
                    if i % 50 == 0:
                        pbar.set_postfix(
                            ok=stats['ok'],
                            rej=stats['reject_sev']+stats['reject_len']+stats['reject_refusal'],
                            err=stats['error'], cache=len(cache),
                        )
                    if i % SAVE_EVERY == 0:
                        pickle.dump(cache, open(CACHE_PATH, 'wb'))
        finally:
            # Always flush cache — even on Ctrl+C / cancel / exception
            pickle.dump(cache, open(CACHE_PATH, 'wb'))
    print(f'\nstats: {stats}')
    return out

# Quick sanity call — inspect before running the full batch.
sample = rows[0]['messages'][1]['content']
print('\nORIG:', sample[:300], '...')
print()
_result = await run_batch([sample], desc='sanity')
print('NEW :', _result[0][:300], '...')

In [ ]:
# Full paraphrase pass on TRAIN ONLY (eval already set aside).
# Safe to interrupt and resume — cache persists to disk every 500 items.

PARAPHRASE = True   # flip to False to skip

if PARAPHRASE and LLM_HOST:
    inputs = [r['messages'][1]['content'] for r in rows]
    already = sum(1 for t in inputs if hashlib.md5(t.encode()).hexdigest() in cache)
    print(f'paraphrasing {len(inputs):,} TRAIN assistant responses (cached: {already:,})')
    t0 = time.time()
    new_texts = await run_batch(inputs)
    print(f'done in {time.time()-t0:.1f}s')
    para_rows = []
    for r, new in zip(rows, new_texts):
        nr = {'messages': [r['messages'][0], {'role': 'assistant', 'content': new}]}
        para_rows.append(nr)
    write_jsonl(CKPT_DIR / '05_train_paraphrased.jsonl', para_rows)
    train_rows = para_rows
    rows = para_rows
    # Quick sanity on paraphrase quality
    kept = sum(1 for a, b in zip(inputs, new_texts) if a != b)
    print(f'paraphrase changed {kept:,}/{len(inputs):,} outputs ({kept/len(inputs)*100:.1f}%)')
else:
    train_rows = rows
    print('Skipped paraphrase (PARAPHRASE=False or LLM_HOST unset).')

paraphrasing 149,946 TRAIN assistant responses (cached: 149,946)


paraphrase:  22%|██▏       | 32500/149946 [00:49<02:47, 699.82req/s, cache=144949, err=0, ok=0, rej=0]

## 9. Final counts (split already done in stage 7)

In [ ]:
# Split was done in stage 7 (before paraphrase). Just report.
print(f'train={len(train_rows):,}   eval={len(eval_rows):,}')
print('train sev:', dict(Counter(severity(r['messages'][1]['content']) for r in train_rows)))
print('eval  sev:', dict(Counter(severity(r['messages'][1]['content']) for r in eval_rows)))

## 10. Write outputs

In [ ]:
write_jsonl(OUT_TRAIN, train_rows)
write_jsonl(OUT_EVAL, eval_rows)

## 11. Post-write diagnostics (verify it actually improved)

In [ ]:
def diag(path):
    rows = list(iter_jsonl(path))
    a_exact = Counter(r['messages'][1]['content'] for r in rows)
    u_exact = Counter(r['messages'][0]['content'] for r in rows)
    a_skel = Counter(skeleton(r['messages'][1]['content'], 100) for r in rows)
    sev = Counter(severity(r['messages'][1]['content']) for r in rows)
    print(f'\n{path.name}   n={len(rows):,}')
    print(f'  unique user/asst : {len(u_exact):,} / {len(a_exact):,}')
    print(f'  asst skeletons   : {len(a_skel):,}')
    print(f'  severity         : {dict(sev)}')
    print(f'  top asst skel    : {a_skel.most_common(1)[0][1]} copies of  {a_skel.most_common(1)[0][0][:80]!r}')

diag(OUT_TRAIN)
diag(OUT_EVAL)